In [1]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores.chroma import Chroma

from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [2]:
caminhos = [
    "files/apostila.pdf",
    "files/LLM.pdf"
]

paginas = []
for caminho in caminhos:
    loader = PyPDFLoader(caminho)
    paginas.extend(loader.load())


recur_split = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", ""]
)

documents = recur_split.split_documents(paginas)

for i, doc in enumerate(documents):
    doc.metadata['source'] = doc.metadata['source'].replace('arquivos/', '')
    doc.metadata['doc_id'] = i


/Users/luizfelipew/Documents/git/AI/Exploring-LLMs/agents/agentes-ai-apps/.venv/lib/python3.13/site-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 42 0 (offset 0)
Ignoring wrong pointing object 50 0 (offset 0)
Ignoring wrong pointing object 52 0 (offset 0)
Ignoring wrong pointing object 54 0 (offset 0)
Ignoring wrong pointing object 56 0 (offset 0)
Ignoring wrong pointing object 58 0 (offset 0)
Ignoring wrong pointing object 70 0 (offset 0)
Ignoring wrong pointing object 72 0 (offset 0)
Ignoring wrong 

In [3]:
from langchain_chroma import Chroma
diretorio = "arquivo/chat_retrieval_db"

embeddings_model = OpenAIEmbeddings()
vectordb = Chroma.from_documents(
    documents=documents,
    embedding=embeddings_model,
    persist_directory=diretorio
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [4]:
from langchain_openai.chat_models import ChatOpenAI

chat = ChatOpenAI(model="gpt-4o-mini")

In [5]:
from langchain.chains.retrieval_qa.base import RetrievalQA

chat_chain = RetrievalQA.from_chain_type(
    llm=chat,
    retriever=vectordb.as_retriever(search="mmr")
)

In [7]:
pergunta = "O que é o Hugging Face e como faço para acessá-lo"
chat_chain.invoke({'query': pergunta})

{'query': 'O que é o Hugging Face e como faço para acessá-lo',
 'result': 'O Hugging Face é uma comunidade e uma plataforma que fornece uma vasta coleção de modelos de aprendizado de máquina, especialmente voltados para processamento de linguagem natural (NLP). Ele reúne centenas de milhares de modelos contribuídos por usuários e desenvolvedores, que podem ser usados para diversas tarefas, como geração de texto, resumo e classificação.\n\nPara acessar o Hugging Face, você pode visitar o site oficial (huggingface.co), onde encontrará uma interface amigável para explorar e baixar modelos. Além disso, você pode usar a biblioteca Transformers da Hugging Face em seu código Python, que facilita a integração e o uso desses modelos em seus projetos. Para começar, você precisará instalar a biblioteca usando pip:\n\n```bash\npip install transformers\n```\n\nApós a instalação, você pode carregar e usar modelos com algumas linhas de código simples.'}

In [10]:
from langchain.prompts import PromptTemplate

chain_prompt = PromptTemplate.from_template(
"""Utilize o contexto fornecido para responder a pergunta ao final.
Se você não sabe a resposta, apenas diga que não sabe e não invente uma resposta.
Utilize três frases no máximo, mantenha a resposta cancisa

Contexto: {context}

Pergunta: {question}

Resposta
"""
)

In [19]:
chat_chain = RetrievalQA.from_chain_type(
    llm=chat,
    retriever=vectordb.as_retriever(search="mmr"),
    chain_type_kwargs={"prompt": chain_prompt},
    return_source_documents=True
)

In [20]:
pergunta = "O que é o Hugging Face e como faço para acessá-lo"
resposta = chat_chain.invoke({'query': pergunta})
print(resposta['result'])

Hugging Face é uma comunidade de código aberto que oferece uma vasta coleção de modelos de linguagem, como geração de texto, resumo e classificação. Você pode acessá-lo através do site oficial da Hugging Face, onde pode explorar e baixar modelos disponíveis. Além disso, a biblioteca Transformers da Hugging Face facilita a integração desses modelos em projetos de Python.


In [21]:
resposta["source_documents"]

[Document(page_content='Atualmente, requer um pouco mais de esforço para pegar um modelo de código aberto e começar a usá-lo, mas o progresso está ocorrendo muito rapidamente para torná-los mais acessíveis aos usuários. Na Databricks, por exemplo, fizemos melhorias em frameworks de código aberto como o MLflow para tornar muito fácil para alguém com um pouco de experiência em Python pegar qualquer modelo transformador da Hugging Face e usá-lo como um objeto Python. Muitas vezes, você pode encontrar um modelo de código aberto que resolve seu problema específico e que é várias ordens de grandeza menor que o ChatGPT, permitindo que você traga o modelo para seu ambiente e hospede-o você mesmo. Isso significa que você pode manter os dados sob seu controle para preocupações com privacidade e governança, além de gerenciar seus custos. Outra grande vantagem de usar modelos de código aberto é a capacidade de ajustá-los aos seus próprios dados', metadata={'doc_id': 75, 'page': 6, 'source': 'files

In [22]:
from langchain.globals import set_debug

set_debug(True)

pergunta = "O que é Hugging Face  e como eu faço para acessá-lo"
resposta = chat_chain.invoke({'query': pergunta})

set_debug(False)

[chain/start] [chain:RetrievalQA] Entering Chain run with input:
{
  "query": "O que é Hugging Face  e como eu faço para acessá-lo"
}
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain] Entering Chain run with input:
[inputs]
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain] Entering Chain run with input:
{
  "question": "O que é Hugging Face  e como eu faço para acessá-lo",
  "context": "Atualmente, requer um pouco mais de esforço para pegar um modelo de código aberto e começar a usá-lo, mas o progresso está ocorrendo muito rapidamente para torná-los mais acessíveis aos usuários. Na Databricks, por exemplo, fizemos melhorias em frameworks de código aberto como o MLflow para tornar muito fácil para alguém com um pouco de experiência em Python pegar qualquer modelo transformador da Hugging Face e usá-lo como um objeto Python. Muitas vezes, você pode encontrar um modelo de código aberto que resolve seu problema específico e que é várias ordens de gr

In [24]:
chat_chain = RetrievalQA.from_chain_type(
    llm=chat,
    retriever=vectordb.as_retriever(search="mmr"),
    chain_type="refine"
)

pergunta = "O que é o Hugging Face e como faço para acessá-lo"
resposta = chat_chain.invoke({'query': pergunta})
print(resposta['result'])

Hugging Face é uma plataforma que oferece uma biblioteca de modelos de aprendizado de máquina, com foco especial em modelos de processamento de linguagem natural (NLP). Este repositório abriga uma vasta coleção de modelos pré-treinados que podem ser utilizados em diversas tarefas, como tradução, resumo, geração de texto e classificação. A Hugging Face também conta com uma comunidade ativa que promove o compartilhamento e a implementação de modelos de código aberto.

A principal vantagem dos modelos de código aberto, como os oferecidos pela Hugging Face, é que eles permitem uma maior flexibilidade e controle sobre a implementação. Além disso, devido aos recursos computacionais necessários, serviços proprietários geralmente não são gratuitos além de um uso muito limitado, o que torna o custo um fator importante ao aplicá-los em grande escala. Resumindo, serviços proprietários são ótimos para tarefas muito complexas, desde que você esteja disposto a compartilhar seus dados com terceiros e